# Forecast VAR, Source-Aware Edition

**Episode 2 story:** *I built an AI to predict the 2026 World Cup, then forced it to prove its sources.*

This notebook is the end-to-end experiment behind the YouTube episode. The drama is deliberately sports-shaped, but the lesson is agent engineering:

> A forecast agent is only useful if it can separate **football uncertainty** from **AI hallucination**.

In football, VAR means Video Assistant Referee. Here, Forecast VAR is the review-booth metaphor for AI forecasting: validate inputs, audit evidence, check claims, and refuse unsupported certainty before a prediction is presented.

The episode starts like a pundit panel: everyone wants a winner, every model has favourites, and a single upset can ruin the bracket. Then Forecast VAR steps in like a video-review booth. Before the agent is allowed to say who might win, it must prove five things:

1. the tournament field and feature table are internally valid,
2. the source coverage is disclosed,
3. forecast numbers come from deterministic tools rather than LLM vibes,
4. Monte Carlo probabilities are explained as simulations, not prophecy,
5. every final answer passes claim-level evaluation.

The workflow is:

1. Refresh a local evidence index.
2. Run forecast pre-flight validation.
3. Inspect source coverage.
4. Compare model output with a sample de-vig market baseline.
5. Run a whole-tournament Monte Carlo.
6. Demonstrate a rolling forecast after a completed-result state.
7. Evaluate the baseline and grounded agents.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from forecast_var.config import DEFAULT_OPENAI_MODEL
from forecast_var.tools import (
    refresh_evidence_index,
    preflight_forecast_context,
    source_coverage_report,
    forecast_match_with_context,
    simulate_tournament,
    rolling_group_forecast,
)
from forecast_var.eval_harness import evaluate
from forecast_var.mock_agent import run_baseline_mock, run_grounded_mock

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Default live OpenAI model:", DEFAULT_OPENAI_MODEL)

## 0. Who predicts: `gpt-5.4-nano` or Monte Carlo?

Both appear in the project, but they do different jobs.

```text
gpt-5.4-nano
= live agent brain / router / tool caller / explainer

Python Monte Carlo simulator
= forecasting engine that generates probabilities through deterministic tools
```

In live mode, `gpt-5.4-nano` is the presenter in the studio. It reads the question, loads the right skills, calls MCP tools, and explains the output with caveats. The Monte Carlo simulator is the stats department. It repeatedly samples group and knockout outcomes, then counts how often each team becomes champion, finalist, or semifinalist.

This separation is the whole point of the episode. If the language model invents probabilities, the demo is flashy but unauditable. If the language model calls a tool, the numeric forecast can be reproduced when the code, bundled inputs, parameters, and simulator seed are the same. The live LLM API response itself is not guaranteed to be bit-for-bit deterministic, even with low temperature or seed controls. The tool boundary makes the prediction-producing part reproducible; the tool trace, citations, structured claims, and verifier make the LLM explanation auditable.

```text
User question
   ↓
gpt-5.4-nano Forecast VAR agent
   ↓
MCP tool call: simulate_tournament
   ↓
Python Monte Carlo engine
   ↓
probabilities
   ↓
gpt-5.4-nano explanation + citations + uncertainty
   ↓
claim verifier + eval harness
```

## 1. Build the local evidence index

The project does not use live APIs by default. Instead, it builds a transparent JSONL evidence index from bundled source cards, team features, sample market rows, and curated notes.

In [ ]:
refresh = refresh_evidence_index()
refresh

## 2. Forecast pre-flight validation

Before the agent is allowed to forecast, it must prove that the tournament field and model inputs are internally consistent.

In [ ]:
preflight = preflight_forecast_context()
{
    "ready_for_forecast": preflight["ready_for_forecast"],
    "group_count": preflight["group_count"],
    "team_count": preflight["team_count"],
    "feature_rows": preflight["feature_rows"],
    "evidence_document_count": preflight["evidence_document_count"],
    "warnings": preflight["warnings"],
}

![Pre-flight readiness](../figures/preflight_readiness.png)

## 3. Source coverage report

A forecast should disclose its source coverage. Here the agent can distinguish official tournament structure from demo priors, sample market baselines, adapter slots, and missing live injury/lineup data.

In [ ]:
coverage = source_coverage_report()
{
    "coverage": coverage["coverage"],
    "status_counts": coverage["status_counts"],
    "evidence_document_counts": coverage["evidence_document_counts"],
    "warnings": coverage["warnings"],
}

![Source coverage](../figures/source_coverage_status.png)

## 4. Model vs market baseline

Market data should not become betting advice. In Forecast VAR it is a comparison baseline: de-vig the sample odds, compare against the model, disclose margin and timestamp, then show data gaps.

De-vig means removing bookmaker margin, also called vig or overround. The tool converts each decimal odd into a raw implied probability with `1 / odds`, then normalizes the win/draw/win probabilities so they sum to 1. That gives a cleaner sample market-implied baseline for comparison, not a wagering recommendation or a claim about live licensed odds.

In [ ]:
match_context = forecast_match_with_context("USA", "Australia")
{
    "model_probabilities": match_context["forecast"]["probabilities"],
    "market_baseline": match_context["market_baseline"],
    "model_vs_market": match_context["model_vs_market"],
    "data_gaps": match_context["data_gaps"],
}

![Model vs market](../figures/model_vs_market_usa_australia.png)

## 5. Whole-tournament Monte Carlo

The upgraded agent can run a whole-tournament simulation. This is where the sports drama becomes useful for teaching agent design.

A deterministic bracket says: *Team A is stronger, so Team A advances.*

Monte Carlo says: *Team A is stronger, but football has variance.* The engine runs hundreds or thousands of alternate tournaments and then reports how often each team survives the chaos. That is why the final output is a probability distribution, not a prophecy. The simulator is deliberately documented as approximate: it models group advancement and a seeded 32-team knockout, not the official FIFA bracket path.

In [ ]:
mc = simulate_tournament(sims=600, limit=8)
mc["top_teams"]

![Monte Carlo champion probabilities](../figures/monte_carlo_champion_probabilities.png)

## 6. Rolling forecast after a completed-result state

Rolling mode locks completed results before forecasting the remaining path. The example below is intentionally illustrative, not a real 2026 result.

In [ ]:
rolling = rolling_group_forecast(
    "D",
    [{"team_a": "USA", "team_b": "Australia", "team_a_goals": 2, "team_b_goals": 1}],
    sims=800,
)
rolling

## 7. Agent answer examples

The deterministic grounded agent follows the same workflow a live OpenAI/MCP agent is instructed to follow: source search, pre-flight, tool call, typed claims, citations, and claim verification. The live version uses `gpt-5.4-nano` by default, but the offline notebook uses deterministic mock mode so the episode is reproducible without API calls.

For live API runs, treat reproducibility as protocol reproducibility rather than identical prose: preserve the model name, prompt, tool inputs, tool outputs, code version, data snapshot, and simulator seed, then compare the structured answer and claim verification.

In [ ]:
questions = [
    "Give me a source coverage report before using the forecast agent for public predictions.",
    "Compare the model against the sample market baseline for USA vs Australia.",
    "Run a Monte Carlo tournament simulation and show the top champion probabilities.",
    "After USA beat Australia 2-1, run a rolling Group D forecast.",
]

for q in questions:
    ans = run_grounded_mock(q)
    print("QUESTION:", q)
    print("ANSWER:", ans.answer)
    print("TOOLS:", ans.tools_used)
    print("SKILLS:", ans.skills_used)
    print("PROBABILITIES:", ans.probabilities)
    print("-" * 100)

### Live OpenAI path used in the episode examples

The live path keeps the same architecture but swaps the deterministic mock narrator for the OpenAI Agents SDK agent. The default model is `gpt-5.4-nano`.

```bash
export OPENAI_API_KEY="your_key_here"

PYTHONPATH=src python scripts/run_agent.py   "Run a Monte Carlo tournament simulation and show the top champion probabilities."   --mode openai
```

You can still override the model explicitly:

```bash
PYTHONPATH=src python scripts/run_agent.py   "Compare the model against the sample market baseline for USA vs Australia."   --mode openai   --model gpt-5.4-nano
```


## 8. Evaluation harness

The baseline is intentionally careless: no tools, no pre-flight, no citations, no uncertainty discipline. The grounded agent must pass the same cases with explicit source support.

In [ ]:
# The scripts already materialise the evaluation summaries in reports/.
# Reading them here keeps the notebook fast and makes the episode reproducible.
baseline = json.loads((PROJECT_ROOT / "reports/summary_baseline_mock.json").read_text())
grounded = json.loads((PROJECT_ROOT / "reports/summary_grounded_mock.json").read_text())
{"baseline": baseline, "grounded": grounded}

![Evaluation summary](../figures/eval_summary.png)

## Takeaway

The prediction itself is not the point. The agent-engineering lesson is that a credible forecast agent needs a hard boundary between **LLM orchestration** and **statistical forecasting**:

- source coverage disclosure,
- source-specific adapters,
- reproducible evidence indexing,
- seeded deterministic model tools,
- market-baseline comparison without betting advice,
- rolling-state support,
- typed claims and citations,
- evaluation beyond “the answer looked good”.